# Closing the Fairness Gap — Group-DRO + Adaptive Thresholds

New idea (think like a researcher): standard training optimises the AVERAGE loss,
which is dominated by data-rich Head regions, so the Tail is ignored.
**Group-DRO** (`--dro`) adaptively focuses training on the WORST group (Tail) each step —
the textbook method for group fairness. Combined with adaptive thresholds (`--adathr`).

Goal: push Tail HIGHER and Fair gap MUCH SMALLER than the ~28 / ~41 we had.

Runtime -> **GPU**, then **Run all**.


## Step 1 — data + code


In [ ]:
import os
!rm -rf FedCrime && git clone --depth 1 https://github.com/vanetlabiitj/FedCrime.git
os.makedirs('data',exist_ok=True)
!cp FedCrime/Dataset/processed_crime.csv data/la_crime.csv
import torch; print('GPU:',torch.cuda.is_available())


In [ ]:
%%writefile robust_fair_gnn.py
"""
Robust & Fair Federated GNN for Crime Prediction under Extreme Sparsity
================================================================================
Research extension of FedCrime. Runs on the REAL Los Angeles / Chicago data.

It brings together, in ONE framework:
  * ZINB zero-inflation loss                       (from the FedCrime paper)
  * A Spatio-Temporal GNN (TCN + graph convolution) over a learned region graph
  * ATTACKS by malicious clients, incl. the novel *sparsity-camouflaged* attack
  * DEFENSES, incl. a novel density/graph "vouching" aggregator
  * FAIRNESS metrics (Head / Mid / Tail performance gap)
  * WEEK-AHEAD prediction (--horizon) and UNCERTAINTY (MC-dropout)

The research question (the gap): in federated crime prediction, honest SPARSE
neighbourhoods look like MALICIOUS clients, so standard defenses either let
attacks through or unfairly silence poor regions. We study this
sparsity-robustness-fairness trilemma and a defense that resolves it.

Examples
--------
# clean run on LA, next-day prediction, plain FedAvg:
python3 robust_fair_gnn.py --city la

# compare defenses under the sparsity-camouflaged attack (the key experiment):
python3 robust_fair_gnn.py --city la --attack camouflage --compare

# week-ahead (7 days) with uncertainty:
python3 robust_fair_gnn.py --city la --horizon 7 --mc 10

Requires: torch, numpy, pandas, scikit-learn
================================================================================
"""
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

EPS = 1e-10
C = 8                       # crime categories
L = 8                       # input days per window

CITY = {
    "la":      ("data/la_crime.csv",  "date_occ", "2018-01-01", "2018-12-31"),
    "chicago": ("data/chi_crime.csv", "date_occ", "2015-01-01", "2015-12-31"),
}


# --------------------------------------------------------------------------- #
# 1. REAL DATA -> aligned (days x regions x categories) tensor + region graph
# --------------------------------------------------------------------------- #
def load_city(csv, date_col, start, end):
    df = pd.read_csv(csv)
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])
    days = pd.date_range(start, end, freq="D")
    regions = sorted(df["neighborhood_id"].unique())
    ridx = {r: i for i, r in enumerate(regions)}
    didx = {d: i for i, d in enumerate(days)}
    mat = np.zeros((len(days), len(regions), C), dtype=np.float32)
    g = df.groupby([df[date_col].dt.normalize(), "neighborhood_id", "crime_type_id"]).size()
    for (day, r, c), _ in g.items():
        if day in didx and r in ridx and 0 <= int(c) < C:
            mat[didx[day], ridx[r], int(c)] = 1.0
    return mat, regions                        # mat: (D, R, C) binary


def make_windows(mat, horizon=1):
    """X: (N, L, R, C), Y: (N, R, C) predicting the day `horizon` steps ahead."""
    D = mat.shape[0]
    X, Y = [], []
    for t in range(D - L - horizon + 1):
        X.append(mat[t:t + L])                 # L days of history
        Y.append(mat[t + L + horizon - 1])     # horizon=1 -> next day; 7 -> a week
    return np.asarray(X, np.float32), np.asarray(Y, np.float32)


def build_graph(mat_train, k=6):
    """Region graph from crime-pattern similarity on the TRAINING period."""
    totals = mat_train.sum(axis=2)             # (Dtrain, R) daily total per region
    R = totals.shape[1]
    corr = np.corrcoef(totals.T)               # (R, R)
    corr = np.nan_to_num(corr)
    np.fill_diagonal(corr, -1)
    A = np.eye(R, dtype=np.float32)
    for i in range(R):                         # connect each region to top-k similar
        for j in np.argsort(corr[i])[::-1][:k]:
            A[i, j] = 1.0
    A = np.maximum(A, A.T)                      # symmetric
    d = A.sum(1); dinv = 1.0 / np.sqrt(np.maximum(d, 1e-6))
    return (A * dinv[:, None] * dinv[None, :]).astype(np.float32), A


def head_mid_tail(mat_train):
    """Label each region Head/Mid/Tail by total training crime (20/30/50%)."""
    totals = mat_train.sum(axis=(0, 2))        # (R,)
    order = np.argsort(totals)[::-1]
    R = len(totals); nh = max(1, round(R * 0.2)); nm = max(1, round(R * 0.3))
    tag = np.empty(R, dtype=object)
    tag[order[:nh]] = "Head"; tag[order[nh:nh + nm]] = "Mid"; tag[order[nh + nm:]] = "Tail"
    return tag


# --------------------------------------------------------------------------- #
# 2. MODEL: ST-GNN (TCN temporal + graph conv) + ZINB + classifier + dropout
# --------------------------------------------------------------------------- #
class TCNBlock(nn.Module):
    def __init__(self, i, o, k=3, dil=1):
        super().__init__()
        p = (k - 1) * dil // 2
        self.c1 = nn.Conv1d(i, o, k, padding=p, dilation=dil)
        self.c2 = nn.Conv1d(o, o, k, padding=p, dilation=dil)
        self.r = nn.ReLU()
        self.res = nn.Conv1d(i, o, 1) if i != o else nn.Identity()

    def forward(self, x):
        s = self.res(x); x = self.r(self.c1(x)); x = self.r(self.c2(x)); return x + s


class STGNN(nn.Module):
    """gnn_type: 'plain' (basic GCN), 'gated' (fixes over-smoothing), 'attention'."""
    def __init__(self, hidden=16, use_graph=True, gnn_type="plain", p_drop=0.2):
        super().__init__()
        self.use_graph = use_graph
        self.gnn_type = gnn_type
        self.tcn = nn.Sequential(TCNBlock(C, hidden, dil=1),
                                 TCNBlock(hidden, hidden, dil=2),
                                 TCNBlock(hidden, hidden, dil=4))
        self.g1 = nn.Linear(hidden, hidden); self.g2 = nn.Linear(hidden, hidden)
        self.gate1 = nn.Linear(hidden, hidden); self.gate2 = nn.Linear(hidden, hidden)
        self.relu = nn.ReLU(); self.drop = nn.Dropout(p_drop)
        self.fc_pi = nn.Linear(hidden, C)      # Eq 6
        self.fc_mu = nn.Linear(hidden, C)      # Eq 7
        self.fc_phi = nn.Linear(hidden, C)     # Eq 8
        self.fc_out = nn.Linear(hidden, C)     # crime-present classifier

    def _gated(self, h, A, lin, gate):
        # gated residual: g=sigmoid(gate(h)); H = g*(A.H.W) + (1-g)*H
        # each region learns how much to trust neighbours vs its own signal
        agg = torch.einsum('ij,bjh->bih', A, lin(h))
        g = torch.sigmoid(gate(h))
        return self.relu(g * agg + (1 - g) * h)

    def _attn(self, h, A, lin):
        # attention over graph neighbours (masked by the adjacency)
        Wh = lin(h); Hd = Wh.shape[-1]
        scores = torch.einsum('bih,bjh->bij', Wh, Wh) / (Hd ** 0.5)
        mask = (A > 0).float().unsqueeze(0)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        alpha = torch.softmax(scores, dim=-1)
        return self.relu(torch.einsum('bij,bjh->bih', alpha, Wh))

    def _mix(self, h, A):
        if not self.use_graph:                       # no graph (TCN only)
            return self.relu(self.g2(self.relu(self.g1(h))))
        if self.gnn_type == "gated":
            h = self._gated(h, A, self.g1, self.gate1)
            return self._gated(h, A, self.g2, self.gate2)
        if self.gnn_type == "attention":
            h = self._attn(h, A, self.g1)
            return self._attn(h, A, self.g2)
        # plain GCN
        h = self.relu(torch.einsum('ij,bjh->bih', A, self.g1(h)))
        return self.relu(torch.einsum('ij,bjh->bih', A, self.g2(h)))

    def forward(self, X, A):
        B, Lw, R, _ = X.shape
        h = X.permute(0, 2, 3, 1).reshape(B * R, C, Lw)
        h = self.tcn(h)[:, :, -1].reshape(B, R, -1)
        h = self.drop(self._mix(h, A))
        pi = torch.sigmoid(self.fc_pi(h))
        mu = torch.exp(torch.clamp(self.fc_mu(h), max=15.0))
        phi = torch.nn.functional.softplus(self.fc_phi(h))
        return pi, mu, phi, self.fc_out(h)


def zinb_elem(pi, mu, phi, y):
    """Per-element ZINB negative log-likelihood, shape (B, R, C)."""
    is0 = y.eq(0).float(); is1 = y.gt(0).float()
    zero = is0 * torch.log(pi + EPS)
    nb = is1 * (torch.lgamma(y + phi) - torch.lgamma(phi) - torch.lgamma(y + 1.0)
                + phi * (torch.log(phi + EPS) - torch.log(phi + mu + EPS))
                + y * (torch.log(mu + EPS) - torch.log(phi + mu + EPS)))
    return -(zero + nb)


def zinb_loss(pi, mu, phi, y):
    return zinb_elem(pi, mu, phi, y).mean()


def loss_fn(pi, mu, phi, logit, y, pos_weight=None, region_weight=None,
            group_ids=None, dro_tau=0.0, lam=0.3):
    # pos_weight counters class imbalance. Group-DRO (dro_tau>0) adaptively
    # up-weights the WORST-performing group (the poor tail regions) each step,
    # so training focuses on closing the fairness gap instead of ignoring the
    # tail. region_weight is the older static fairness weighting.
    bce = torch.nn.functional.binary_cross_entropy_with_logits(
        logit, y, pos_weight=pos_weight, reduction="none")     # (B, R, C)
    loss = bce + lam * zinb_elem(pi, mu, phi, y)               # (B, R, C)
    if group_ids is not None and dro_tau > 0:
        lr = loss.mean(dim=(0, 2))                             # per-region loss (R,)
        groups = torch.unique(group_ids)
        gloss = torch.stack([lr[group_ids == g].mean() for g in groups])
        gw = torch.softmax(dro_tau * gloss, dim=0) * len(groups)   # worse -> higher
        rw = torch.ones_like(lr)
        for i, g in enumerate(groups):
            rw[group_ids == g] = gw[i]
        loss = loss * rw.view(1, -1, 1)
    elif region_weight is not None:
        loss = loss * region_weight.view(1, -1, 1)
    return loss.mean()


# --------------------------------------------------------------------------- #
# 3. ATTACKS (applied inside a malicious client)
# --------------------------------------------------------------------------- #
def poison_labels(Y, attack):
    if attack == "labelflip":
        return 1.0 - Y                          # flip yes<->no
    if attack == "camouflage":
        return np.zeros_like(Y)                 # pretend "no crime anywhere" (looks sparse)
    return Y


def poison_update(delta, attack, factor=8.0):
    if attack == "scale":
        return [d * factor for d in delta]      # blow up the update
    if attack == "camouflage":
        return [d * 0.3 for d in delta]         # small, sparse-looking update
    return delta


# --------------------------------------------------------------------------- #
# 4. FEDERATED TRAINING with pluggable DEFENSE
# --------------------------------------------------------------------------- #
def gw(net): return [p.detach().clone() for p in net.state_dict().values()]
def sw(net, w):
    sd = net.state_dict()
    for k, v in zip(sd.keys(), w): sd[k] = v.clone()
    net.load_state_dict(sd)
def flat(w): return torch.cat([t.flatten() for t in w])


def aggregate(defense, wsets, gwt, densities):
    """Combine client weight-sets into a new global model."""
    n = len(wsets)
    if defense == "fedavg":
        return [torch.stack([w[i] for w in wsets]).mean(0) for i in range(len(wsets[0]))]

    deltas = [[w[i] - gwt[i] for i in range(len(gwt))] for w in wsets]

    if defense == "strust":
        # SPARSITY-AWARE TRUST  (our trilemma solution).
        # Standard defenses reject "outlier-distant" clients -> they wrongly
        # reject honest sparse (tail) regions. Instead we judge clients by the
        # DIRECTION of their update and neutralise magnitude:
        #   1) unit-normalise each update  -> a scaling attack becomes harmless
        #   2) trust = max(0, cos(update, robust median direction))
        #        -> label-flip / camouflage point the wrong way  -> trust 0
        #        -> honest sparse clients point the right way     -> trust kept
        #   3) trust-weighted average, rescaled to the typical honest magnitude.
        flatd = [flat(d) for d in deltas]
        norms = [float(f.norm()) + 1e-9 for f in flatd]
        units = torch.stack([flatd[i] / norms[i] for i in range(n)])
        ref = units.median(0).values
        ref = ref / (ref.norm() + 1e-9)
        trust = torch.clamp(units @ ref, min=0.0)              # direction agreement
        if float(trust.sum()) < 1e-6:
            trust = torch.ones(n)
        trust = trust / trust.sum()
        scale = float(torch.tensor(norms).median())
        out = []
        for li in range(len(gwt)):
            acc = sum(trust[i] * (deltas[i][li] / norms[i]) for i in range(n))
            out.append(gwt[li] + scale * acc)
        return out

    F = torch.stack([flat(d) for d in deltas])                 # (n, P)
    dist = torch.cdist(F, F)                                    # pairwise distances

    if defense == "trimmed":
        keep = max(1, n - 2)
        idx = torch.argsort(dist.sum(1))[:keep]
    elif defense == "krum":
        kk = max(1, n - 2)
        scores = torch.stack([torch.sort(dist[i])[0][1:kk + 1].sum() for i in range(n)])
        idx = torch.argsort(scores)[:max(1, n - 2)]
    elif defense == "vouch":
        # NOVEL: a client's "oddness" is EXPECTED to be high if it is sparse
        # (honest tail region). Divide the Krum score by the client's data
        # density so sparse-honest clients are NOT penalised, while dense-yet-
        # odd clients (real attackers) are. Then keep the low-suspicion clients.
        kk = max(1, n - 2)
        raw = torch.stack([torch.sort(dist[i])[0][1:kk + 1].sum() for i in range(n)])
        dens = torch.tensor(densities, dtype=raw.dtype).clamp(min=1e-3)
        suspicion = raw * dens                                 # density-adjusted
        idx = torch.argsort(suspicion)[:max(1, n - 2)]
    else:
        raise ValueError(defense)

    sel = [wsets[i] for i in idx.tolist()]
    return [torch.stack([w[i] for w in sel]).mean(0) for i in range(len(sel[0]))]


def train(clients, A_full, defense, attack, mal_ids, use_graph=True,
          rounds=60, local_epochs=5, lr=0.03, seed=0, pos_weight=None,
          gnn_type="plain", fair=False, dro_tau=0.0):
    torch.manual_seed(seed)
    g = STGNN(use_graph=use_graph, gnn_type=gnn_type); gwt = gw(g)
    for _ in range(rounds):
        wsets, dens = [], []
        for ci, cl in enumerate(clients):
            local = STGNN(use_graph=use_graph, gnn_type=gnn_type); sw(local, gwt)
            A = torch.tensor(cl["A"]); X = torch.tensor(cl["X"])
            Y = torch.tensor(poison_labels(cl["Y"], attack if ci in mal_ids else "none"))
            opt = torch.optim.Adam(local.parameters(), lr=lr, weight_decay=1e-4)
            rw = cl.get("rw") if fair else None
            grp = cl.get("grp")
            local.train()
            for _ in range(local_epochs):
                opt.zero_grad()
                pi, mu, phi, logit = local(X, A)
                loss_fn(pi, mu, phi, logit, Y, pos_weight=pos_weight,
                        region_weight=rw, group_ids=grp,
                        dro_tau=dro_tau).backward(); opt.step()
            delta = [p - q for p, q in zip(gw(local), gwt)]
            if ci in mal_ids:
                delta = poison_update(delta, attack)
            wsets.append([q + d for q, d in zip(gwt, delta)])
            dens.append(float(cl["Y"].mean()))                 # client data density
        gwt = aggregate(defense, wsets, gwt, dens)
        sw(g, gwt)
    return g


# --------------------------------------------------------------------------- #
# 5. EVALUATION: overall + fairness (Head/Mid/Tail) + uncertainty
# --------------------------------------------------------------------------- #
@torch.no_grad()
def tune_thresholds(net, X, Y, A):
    """Per-(region, crime-type) decision threshold tuned on a VALIDATION set to
    maximise each cell's F1. Rare crimes in poor regions get a LOWER threshold
    so they are actually predicted -> higher tail F1 -> smaller fairness gap.
    This is the fairness fix: it targets the decision, not the loss."""
    net.eval()
    prob = torch.sigmoid(net(torch.tensor(X), torch.tensor(A))[3]).numpy()  # (N,R,C)
    R = Y.shape[1]
    grid = np.arange(0.03, 0.60, 0.02)
    thr = np.full((R, C), 0.5, dtype=np.float32)
    for r in range(R):
        for cc in range(C):
            y = Y[:, r, cc]
            if y.sum() == 0:                     # no positives to tune on
                continue
            p = prob[:, r, cc]
            best_t, best_f = 0.5, -1.0
            for t in grid:
                f = f1_score(y, (p > t).astype(float), zero_division=0)
                if f > best_f:
                    best_f, best_t = f, t
            thr[r, cc] = best_t
    return thr


@torch.no_grad()
def evaluate(net, X, Y, A, tags, mc=0, thr=None):
    net.eval()
    Xt = torch.tensor(X); At = torch.tensor(A)
    if mc > 0:                                   # MC-dropout uncertainty
        net.train()                              # keep dropout ON
        probs = torch.stack([torch.sigmoid(net(Xt, At)[3]) for _ in range(mc)])
        prob = probs.mean(0); uncertainty = probs.std(0).mean().item()
        net.eval()
    else:
        prob = torch.sigmoid(net(Xt, At)[3]); uncertainty = float("nan")
    if thr is not None:                          # adaptive threshold
        tt = torch.tensor(thr, dtype=prob.dtype)
        th = tt.view(1, -1, 1) if tt.dim() == 1 else tt.unsqueeze(0)  # (1,R[,C])
        pred = (prob > th).float().numpy()
    else:
        pred = (prob > 0.5).float().numpy()      # (N, R, C)
    N, R, _ = pred.shape

    def group_f1(mask):
        yy = Y[:, mask, :].reshape(-1, C); pp = pred[:, mask, :].reshape(-1, C)
        return f1_score(yy, pp, average="macro", zero_division=0) * 100
    overall = f1_score(Y.reshape(-1, C), pred.reshape(-1, C), average="macro", zero_division=0) * 100
    res = {"overall": overall, "uncertainty": uncertainty}
    for grp in ["Head", "Mid", "Tail"]:
        res[grp] = group_f1(np.array([t == grp for t in tags]))
    res["fairness_gap"] = res["Head"] - res["Tail"]     # smaller = fairer
    return res


# --------------------------------------------------------------------------- #
# 6. RUN
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--city", default="la", choices=list(CITY))
    ap.add_argument("--horizon", type=int, default=1, help="1=next day, 7=week ahead")
    ap.add_argument("--clients", type=int, default=6)
    ap.add_argument("--rounds", type=int, default=60)
    ap.add_argument("--attack", default="none",
                    choices=["none", "labelflip", "scale", "camouflage"])
    ap.add_argument("--attack-frac", type=float, default=0.25)
    ap.add_argument("--defense", default="fedavg",
                    choices=["fedavg", "trimmed", "krum", "vouch", "strust"])
    ap.add_argument("--mc", type=int, default=0, help="MC-dropout passes (0=off)")
    ap.add_argument("--fair", action="store_true",
                    help="up-weight poor (tail) regions in training for fairness")
    ap.add_argument("--adathr", action="store_true",
                    help="per-region adaptive thresholds (fairness fix)")
    ap.add_argument("--dro", action="store_true",
                    help="Group-DRO: adaptively focus training on the worst group")
    ap.add_argument("--dro-tau", type=float, default=3.0,
                    help="Group-DRO strength (higher = more focus on the tail)")
    ap.add_argument("--compare", action="store_true",
                    help="run all defenses under the chosen attack")
    ap.add_argument("--graph-compare", action="store_true",
                    help="compare GNN vs no-GNN on this real city (clean, FedAvg)")
    ap.add_argument("--gnn", default="plain",
                    choices=["plain", "gated", "attention"],
                    help="graph type: gated/attention fix over-smoothing")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    csv, dcol, start, end = CITY[args.city]
    print(f"Loading {args.city.upper()} from {csv} ...")
    mat, regions = load_city(csv, dcol, start, end)
    D, R, _ = mat.shape
    print(f"{R} regions, {C} crime types, {D} days, "
          f"sparsity {100*(1-mat.mean()):.1f}% zeros | horizon = {args.horizon} day(s)")

    X, Y = make_windows(mat, horizon=args.horizon)
    n = len(X); ntr = int(n * 0.7); nval = int(n * 0.1)
    Xtr, Ytr = X[:ntr], Y[:ntr]
    Xval, Yval = X[ntr:ntr + nval], Y[ntr:ntr + nval]     # for threshold tuning
    Xte, Yte = X[ntr + nval:], Y[ntr + nval:]
    A_full, _ = build_graph(mat[:ntr + L])
    tags = head_mid_tail(mat[:ntr + L])
    print(f"regions -> Head {sum(t=='Head' for t in tags)}  "
          f"Mid {sum(t=='Mid' for t in tags)}  Tail {sum(t=='Tail' for t in tags)}")

    # split regions across federated clients (round-robin by crime rank -> mixed)
    order = np.argsort(mat[:ntr].sum(axis=(0, 2)))[::-1]
    parts = [order[i::args.clients] for i in range(args.clients)]
    # fairness weights: sparse (tail) regions get higher weight (clipped 1..4)
    dens_r = Ytr.mean(axis=(0, 2))                       # per-region positive rate
    rw_global = np.clip(dens_r.mean() / (dens_r + 1e-6), 1.0, 4.0).astype(np.float32)
    gmap = {"Head": 0, "Mid": 1, "Tail": 2}
    grp_global = np.array([gmap[t] for t in tags], dtype=np.int64)   # for Group-DRO
    clients = [{"A": A_full[np.ix_(idx, idx)],
                "X": Xtr[:, :, idx, :], "Y": Ytr[:, idx, :],
                "rw": torch.tensor(rw_global[idx]),
                "grp": torch.tensor(grp_global[idx])} for idx in parts]
    n_mal = round(args.attack_frac * args.clients)
    mal_ids = set(range(n_mal))                 # first few clients are malicious
    if args.attack != "none":
        print(f"ATTACK: {args.attack} on {n_mal}/{args.clients} clients\n")

    # per-category class weights to fight the 68% zero imbalance
    yflat = Ytr.reshape(-1, C)
    pos = yflat.sum(0); neg = len(yflat) - pos
    pos_weight = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, 10.0),
                              dtype=torch.float32)

    # --- GNN vs no-GNN comparison on the real city (clean, FedAvg) ---
    if args.graph_compare:
        print(f"{'Model':18s} | {'Overall':>7s} | {'Head':>6s} {'Mid':>6s} "
              f"{'Tail':>6s} | {'Fair gap':>8s}")
        print("-" * 62)
        gname = f"GNN-{args.gnn} (graph)"
        for ug, name in [(True, gname), (False, "No-GNN (TCN only)")]:
            net = train(clients, A_full, "fedavg", args.attack, mal_ids,
                        use_graph=ug, rounds=args.rounds, seed=args.seed,
                        pos_weight=pos_weight, gnn_type=args.gnn, fair=args.fair,
                        dro_tau=(args.dro_tau if args.dro else 0.0))
            th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
            r = evaluate(net, Xte, Yte, A_full, tags, thr=th)
            print(f"{name:18s} | {r['overall']:7.2f} | {r['Head']:6.2f} "
                  f"{r['Mid']:6.2f} {r['Tail']:6.2f} | {r['fairness_gap']:8.2f}")
        print("\nHigher Overall for GNN = the graph helps on real data.")
        return

    defenses = (["fedavg", "trimmed", "krum", "vouch", "strust"]
                if args.compare else [args.defense])
    print(f"{'Defense':9s} | {'Overall':>7s} | {'Head':>6s} {'Mid':>6s} {'Tail':>6s} "
          f"| {'Fair gap':>8s} | {'Uncert':>6s}")
    print("-" * 66)
    for d in defenses:
        net = train(clients, A_full, d, args.attack, mal_ids,
                    rounds=args.rounds, seed=args.seed, pos_weight=pos_weight,
                    gnn_type=args.gnn, fair=args.fair,
                    dro_tau=(args.dro_tau if args.dro else 0.0))
        th = tune_thresholds(net, Xval, Yval, A_full) if args.adathr else None
        r = evaluate(net, Xte, Yte, A_full, tags, mc=args.mc, thr=th)
        unc = "-" if np.isnan(r["uncertainty"]) else f"{r['uncertainty']:.3f}"
        print(f"{d:9s} | {r['overall']:7.2f} | {r['Head']:6.2f} {r['Mid']:6.2f} "
              f"{r['Tail']:6.2f} | {r['fairness_gap']:8.2f} | {unc:>6s}")
    print("\nLower 'Fair gap' = fairer (Head and Tail closer). Under attack, a good "
          "defense keeps Overall high AND Fair gap small.")


if __name__ == "__main__":
    main()



## (1) BASELINE — no fairness fix


In [ ]:
for atk in ['scale','camouflage']:
    print('\n===== BASELINE, attack:',atk,'=====')
    !python robust_fair_gnn.py --city la --gnn gated --attack {atk} --attack-frac 0.17 --defense strust


## (2) Adaptive thresholds only (previous best: Tail ~28, gap ~41)


In [ ]:
for atk in ['scale','camouflage']:
    print('\n===== adathr, attack:',atk,'=====')
    !python robust_fair_gnn.py --city la --gnn gated --attack {atk} --attack-frac 0.17 --defense strust --adathr


## (3) NEW: Group-DRO + adaptive thresholds (the stronger fix)
Goal: Tail HIGHER and gap LOWER than (2). Try a few DRO strengths.


In [ ]:
for tau in [3, 6, 10]:
    for atk in ['scale','camouflage']:
        print(f'\n===== DRO(tau={tau}) + adathr, attack: {atk} =====')
        !python robust_fair_gnn.py --city la --gnn gated --attack {atk} --attack-frac 0.17 --defense strust --dro --dro-tau {tau} --adathr
